In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch
import torch.nn as nn
import torch.optim as optim

# -----------------------------------------------------
# 1. Load Data and Define Base Columns
# -----------------------------------------------------
df = pd.read_csv("ratios_for_ml.csv")

# Ensure date is in datetime format and set index to preserve original order
df['enddate'] = pd.to_datetime(df['enddate'])
df = df.set_index(df.index.values)

# Master list of all numerical features and targets used in modeling
master_cols = [
    'current_ratio', 'debt_to_equity', 'roe',
    'debt_to_assets', 'cash_to_assets', 'inventory_to_assets', 'receivables_to_assets'
]

# ----------------------------
# 2. Clean and Impute Data
# ----------------------------
for c in master_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df[c] = df[c].fillna(df[c].median())

# -----------------------------------------------------
# 3. Split Data (Preserving Identifiers for Final Output)
# -----------------------------------------------------
# Use all master_cols to split the data
X_master = df[master_cols].values
y_dummy = df['roe'].values # Dummy target for splitting

X_train_idx, X_test_idx, _, _ = train_test_split(
    df.index.values, y_dummy, test_size=0.2, random_state=42
)

# Create train/test dataframes for easy column access
df_train = df.loc[X_train_idx].copy()
df_test = df.loc[X_test_idx].copy()

# Initialize final results DataFrame with identifiers
results_df = df_test[['stock', 'enddate', 'roe', 'debt_to_equity']].copy()


# -----------------------------------------------------
# 4. Model Definition (Same MLP architecture for both)
# -----------------------------------------------------
class MLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

# -----------------------------------------------------
# 5. Model 1: ROE Prediction (Targeting Profitability)
# -----------------------------------------------------
TARGET_ROE = 'roe'
FEATURES_ROE = [c for c in master_cols if c != TARGET_ROE]

# Prepare data
X_train_roe = df_train[FEATURES_ROE].values
X_test_roe = df_test[FEATURES_ROE].values
y_train_roe = df_train[TARGET_ROE].values
y_test_roe = df_test[TARGET_ROE].values

# Scale data
scaler_X_roe = StandardScaler()
scaler_y_roe = StandardScaler()

X_train_s_roe = scaler_X_roe.fit_transform(X_train_roe)
X_test_s_roe = scaler_X_roe.transform(X_test_roe)
y_train_s_roe = scaler_y_roe.fit_transform(y_train_roe.reshape(-1, 1)).ravel()

# Train Model
model_roe = MLP(X_train_s_roe.shape[1])
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model_roe.parameters(), lr=0.01)

X_train_t_roe = torch.tensor(X_train_s_roe, dtype=torch.float32)
y_train_t_roe = torch.tensor(y_train_s_roe, dtype=torch.float32).view(-1, 1)

for epoch in range(50): # Reduced epochs for faster execution
    model_roe.train()
    optimizer.zero_grad()
    preds = model_roe(X_train_t_roe)
    loss = loss_fn(preds, y_train_t_roe)
    loss.backward()
    optimizer.step()

# Predict and Inverse Transform
model_roe.eval()
X_test_t_roe = torch.tensor(X_test_s_roe, dtype=torch.float32)
preds_test_s_roe = model_roe(X_test_t_roe).detach().numpy()
preds_test_roe = scaler_y_roe.inverse_transform(preds_test_s_roe).flatten()

# Save ROE predictions
results_df['Predicted_ROE'] = preds_test_roe
print("--- ROE Prediction Complete ---")


# -----------------------------------------------------
# 6. Model 2: Debt to Equity Prediction (Targeting Risk/Solvency)
# -----------------------------------------------------
TARGET_DE = 'debt_to_equity'
FEATURES_DE = [c for c in master_cols if c != TARGET_DE]

# Prepare data
X_train_de = df_train[FEATURES_DE].values
X_test_de = df_test[FEATURES_DE].values
y_train_de = df_train[TARGET_DE].values
y_test_de = df_test[TARGET_DE].values

# Scale data
scaler_X_de = StandardScaler()
scaler_y_de = StandardScaler()

X_train_s_de = scaler_X_de.fit_transform(X_train_de)
X_test_s_de = scaler_X_de.transform(X_test_de)
y_train_s_de = scaler_y_de.fit_transform(y_train_de.reshape(-1, 1)).ravel()

# Train Model
model_de = MLP(X_train_s_de.shape[1])
optimizer = optim.Adam(model_de.parameters(), lr=0.01)

X_train_t_de = torch.tensor(X_train_s_de, dtype=torch.float32)
y_train_t_de = torch.tensor(y_train_s_de, dtype=torch.float32).view(-1, 1)

for epoch in range(50): # Reduced epochs for faster execution
    model_de.train()
    optimizer.zero_grad()
    preds = model_de(X_train_t_de)
    loss = loss_fn(preds, y_train_t_de)
    loss.backward()
    optimizer.step()

# Predict and Inverse Transform
model_de.eval()
X_test_t_de = torch.tensor(X_test_s_de, dtype=torch.float32)
preds_test_s_de = model_de(X_test_t_de).detach().numpy()
preds_test_de = scaler_y_de.inverse_transform(preds_test_s_de).flatten()

# Save D/E predictions
results_df['Predicted_Debt_to_Equity'] = preds_test_de
print("--- D/E Prediction Complete ---")


# -----------------------------------------------------
# 7. Final Output
# -----------------------------------------------------

# Rename actual columns for clarity
results_df = results_df.rename(columns={'roe': 'Actual_ROE', 
                                        'debt_to_equity': 'Actual_Debt_to_Equity'})

output_filename = "combined_financial_predictions.csv"
results_df.to_csv(output_filename, index=False)

print(f"\nSuccessfully saved combined predictions to {output_filename}")
print("Final Output Columns:")
print(results_df.columns.tolist())
print(results_df.head())

--- ROE Prediction Complete ---
--- D/E Prediction Complete ---

Successfully saved combined predictions to combined_financial_predictions.csv
Final Output Columns:
['stock', 'enddate', 'Actual_ROE', 'Actual_Debt_to_Equity', 'Predicted_ROE', 'Predicted_Debt_to_Equity']
    stock    enddate  Actual_ROE  Actual_Debt_to_Equity  Predicted_ROE  \
832  AMRX 2019-12-31   -1.559920              11.245403       1.571887   
970  APEI 2018-12-31    0.079806               0.000000       3.986048   
96   ACAD 2018-12-31   -0.511799               0.000000       1.806645   
587   ALE 2016-12-31    0.082039               0.723930       0.000178   
450  AHPI 2019-06-30   -0.177428               0.000000       0.186750   

     Predicted_Debt_to_Equity  
832                  1.363609  
970                  0.137761  
96                  -0.310813  
587                 -0.284245  
450                  0.812063  
